In [4]:
# ── Setup (run this cell first) ──────────────────────────────────────────────
import numpy as np
import json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

print(f"Shape  : {matrix.shape}")
print(f"Places : {meta['total_places']}")
reachable = matrix[matrix > 0]
print(f"Min dist (reachable) : {reachable.min()/1000:.2f} km")
print(f"Max dist             : {reachable.max()/1000:.2f} km")
print(f"Avg dist             : {reachable.mean()/1000:.2f} km")
print(f"No-route pairs (-1)  : {int((matrix == -1).sum())}")

Shape  : (50, 50)
Places : 50
Min dist (reachable) : 0.06 km
Max dist             : 84.31 km
Avg dist             : 24.19 km
No-route pairs (-1)  : 0


In [6]:
# ── Styled distance table (8×8 corner) ───────────────────────────────────────
import numpy as np, json, pandas as pd

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

N = 8
labels = [n[:22] for n in names[:N]]
block  = matrix[:N, :N] / 1000
block[block < 0] = float('nan')  # -1 → NaN so it shows blank

df = pd.DataFrame(block, index=labels, columns=labels)

In [ ]:
df

,Painted Rhythm Art Gal,𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞,Royal Halls NSCI - Lux,Shree Siddhivinayak Te,Saffron,Paradox Museum Mumbai,Basilica of Our Lady o,Illusion Adventures
Painted Rhythm Art Gal,0.000,4.256,12.562,6.812,4.561,20.270,3.842,22.692
𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞,2.920,0.000,9.954,5.332,7.481,17.662,1.783,25.472
Royal Halls NSCI - Lux,12.143,10.439,0.000,5.538,16.703,9.666,10.869,34.470
Shree Siddhivinayak Te,11.297,9.594,4.663,0.000,15.858,14.492,10.023,33.625
Saffron,5.291,12.719,21.024,11.905,0.000,28.732,8.881,13.208
Paradox Museum Mumbai,20.699,18.995,9.509,12.094,25.260,0.000,19.425,43.026
Basilica of Our Lady o,4.216,2.799,12.043,10.944,8.281,19.751,0.000,28.930
Illusion Adventures,20.625,23.433,31.739,26.289,13.396,39.446,25.597,0.000


In [9]:
# ── Look up distance between two places ──────────────────────────────────────
import numpy as np, json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

def get_distance(name_a: str, name_b: str):
    def find(q):
        for i, n in enumerate(names):
            if q.lower() in n.lower():
                return i, n
        return None, None
    i, a = find(name_a)
    j, b = find(name_b)
    if i is None: print(f"'{name_a}' not found"); return
    if j is None: print(f"'{name_b}' not found"); return
    fmt = lambda v: f"{v/1000:.2f} km" if v >= 0 else "no route"
    print(f"{a}  →  {b} : {fmt(matrix[i][j])}")
    print(f"{b}  →  {a} : {fmt(matrix[j][i])}")

# ← change these to any place names
get_distance('𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞', 'Basilica of Our Lady o')
get_distance('Bandra', 'Saffron')

𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞 𝐒𝐩𝐨𝐫𝐭𝐬  →  Basilica of Our Lady of the Mount (Bandra, Mumbai) : 1.78 km
Basilica of Our Lady of the Mount (Bandra, Mumbai)  →  𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞 𝐒𝐩𝐨𝐫𝐭𝐬 : 2.80 km
Basilica of Our Lady of the Mount (Bandra, Mumbai)  →  Saffron : 8.28 km
Saffron  →  Basilica of Our Lady of the Mount (Bandra, Mumbai) : 8.88 km


In [10]:
# ── Find nearest N places to a given place ────────────────────────────────────
import numpy as np, json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

def nearest(name: str, top_n: int = 5):
    idx = next((i for i, n in enumerate(names) if name.lower() in n.lower()), None)
    if idx is None:
        print(f"'{name}' not found"); return
    row = matrix[idx].copy()
    row[idx]   = float('inf')
    row[row < 0] = float('inf')
    top = np.argsort(row)[:top_n]
    print(f"Nearest {top_n} to '{names[idx]}':")
    for rank, j in enumerate(top, 1):
        print(f"  {rank}. {names[j][:45]:45}  {row[j]/1000:.2f} km")

# ← change to any place name
nearest('Gateway', top_n=5)

Nearest 5 to 'Gateway Of India Mumbai':
  1. Colaba Market                                  0.52 km
  2. Chhatrapati Shivaji Maharaj Vastu Sangrahalay  1.13 km
  3. National Gallery of Modern Art                 1.25 km
  4. Street Food stalls                             1.41 km
  5. Jehangir Art Gallery                           1.54 km


In [11]:
# ── Print full place list with indices ───────────────────────────────────────
import json

meta  = json.load(open('matrices/mumbai_meta.json'))
names = meta['place_names']
cats  = meta['categories']

print(f"{'Idx':>3}  {'Category':<22}  Place")
print("-" * 70)
for i, (n, c) in enumerate(zip(names, cats)):
    print(f"{i:>3}  {c:<22}  {n}")

Idx  Category                Place
----------------------------------------------------------------------
  0  Art Gallery             Painted Rhythm Art Gallery
  1  Adventure Sports        𝐉𝐮𝐠𝐚𝐚𝐝𝐗𝐭𝐫𝐞𝐦𝐞 𝐀𝐝𝐯𝐞𝐧𝐭𝐮𝐫𝐞 𝐒𝐩𝐨𝐫𝐭𝐬
  2  Point of Interest       Royal Halls NSCI - Luxurious Banquet Hall | Exhibition Hall| Marriage Hall | Corporate Event Hall in Worli, Mumbai
  3  Religious Site          Shree Siddhivinayak Temple
  4  Restaurant              Saffron
  5  Museum                  Paradox Museum Mumbai
  6  Religious Site          Basilica of Our Lady of the Mount (Bandra, Mumbai)
  7  Adventure Sports        Illusion Adventures
  8  Religious Site          Shri Mahalakshmi Devi Temple, Mumbai
  9  Tourist Attraction      Chhatrapati Shivaji Maharaj Vastu Sangrahalaya
 10  Tourist Attraction      Rajabai Clock Tower
 11  Historic & Cultural     Gateway Of India Mumbai
 12  Park & Gardens          BNHS Nature Reserve
 13  Café                    Bombay To Barcelona Library Cafe
 14  Po